# Task 6: MLOps and Deployment Simulation

## 6.1 Model Versioning and MLOps Objective

The objective of Task 6 is to simulate how the traffic-volume regression model developed earlier in Part 3 could move from experimentation toward deployment, monitoring and operational alerting.

This task does **not** train a new prediction model. Instead, it reuses the regression experiments and results from Task 4 and extends them into an MLOps workflow.

Two regression model versions are documented from the earlier experiment:

| Model Version | Algorithm | MAE | R² | Role |
|---|---|---:|---:|---|
| v1 | Linear Regression | 813.77 | 0.7259 | Baseline |
| v2 | Random Forest Regressor | 271.30 | 0.9442 | Deployment candidate |

The Random Forest model is selected as the deployment candidate because it achieved substantially lower prediction error and higher explanatory performance than the Linear Regression baseline.

The Task 6 workflow consists of:

1. **Model versioning** – document the previously evaluated regression model versions and their performance.
2. **MLflow experiment tracking** – reuse the Task 4 MLflow tracking information for parameters, metrics and model artifacts.
3. **Deployment simulation** – expose the Random Forest traffic-volume model through a lightweight FastAPI endpoint.
4. **Monitoring simulation** – compare simulated prediction error with the Task 4 Random Forest baseline MAE of approximately **271.30**.
5. **Alerting** – classify monitoring status as:
   - **PASS / Normal** when monitored MAE is no more than 1.5 times the baseline MAE.
   - **ALERT / Requires investigation** when monitored MAE exceeds that threshold.

The resulting workflow demonstrates the operational lifecycle of the selected model without claiming to represent a production deployment.


## 6.2 MLflow Experiment Tracking Integration

Task 4 already implemented MLflow experiment tracking for the traffic-volume regression models. Task 6 therefore reuses that experiment history rather than training duplicate models.

The existing MLflow records provide evidence of:

- Experiment tracking
- Model parameters
- Evaluation metrics
- Model artifacts
- Comparison between Linear Regression and Random Forest

The Random Forest experiment achieved an MAE of approximately **271.30** and an R² of approximately **0.9442**, making it the preferred deployment candidate.

This section connects Task 6 to the existing Task 4 MLflow tracking database and verifies that the previously completed experiment runs can be retrieved successfully.

In [3]:
# ============================================================
# 6.2 Locate Task 4 MLflow Tracking Database
# ============================================================

from pathlib import Path
import logging

logger = logging.getLogger("task6_mlops")
logger.setLevel(logging.INFO)
logger.propagate = False

for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)

formatter = logging.Formatter(
    "%(asctime)s - %(levelname)s - %(name)s - %(message)s"
)

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)

file_handler = logging.FileHandler(
    "task6_mlops.log",
    mode="w"
)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)


# Candidate locations for the Task 4 MLflow database
candidate_paths = [
    Path("mlflow_tracking.db"),
    Path("../mlflow_tracking.db"),
    Path("../task4_mlflow_experiment_tracking/mlflow_tracking.db"),
    Path("../part3_machine_learning/mlflow_tracking.db")
]

existing_paths = [
    path.resolve()
    for path in candidate_paths
    if path.exists()
]

if existing_paths:
    MLFLOW_DB_PATH = existing_paths[0]

    logger.info(
        "Task 4 MLflow tracking database found: %s",
        MLFLOW_DB_PATH
    )
else:
    MLFLOW_DB_PATH = None

    logger.warning(
        "Task 4 MLflow tracking database was not found "
        "in the expected relative locations."
    )

2026-10-08 01:39:32,265 - INFO - task6_mlops - Task 4 MLflow tracking database found: C:\Users\tanak\Documents\GitHub\smart-city-traffic-capstone\part3_machine_learning\mlflow_tracking.db


### 6.2.1 Retrieve Existing MLflow Experiment Runs

The Task 4 MLflow tracking database was successfully located within the Part 3 project directory.

The existing experiment runs are now retrieved directly from MLflow. This avoids duplicating model training and demonstrates that model performance, parameters and experiment history can be reused across stages of the MLOps workflow.

The retrieved results will also be used to confirm the deployment candidate and establish the baseline MAE for subsequent monitoring.


In [4]:
# ============================================================
# 6.2.1 Retrieve Existing MLflow Experiment Runs
# ============================================================

import mlflow
import pandas as pd

if MLFLOW_DB_PATH is None:
    raise FileNotFoundError(
        "Task 4 MLflow tracking database is unavailable."
    )

# Connect directly to the Task 4 SQLite tracking database
TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH.as_posix()}"

mlflow.set_tracking_uri(TRACKING_URI)

logger.info(
    "MLflow tracking URI configured: %s",
    TRACKING_URI
)

# Retrieve available experiments
experiments = mlflow.search_experiments()

if not experiments:
    raise RuntimeError(
        "No MLflow experiments were found in the tracking database."
    )

logger.info(
    "MLflow experiments found: %d",
    len(experiments)
)

# Retrieve all runs from all experiments
experiment_ids = [
    experiment.experiment_id
    for experiment in experiments
]

runs = mlflow.search_runs(
    experiment_ids=experiment_ids,
    order_by=["start_time DESC"]
)

if runs.empty:
    raise RuntimeError(
        "No MLflow runs were found in the tracking database."
    )

logger.info(
    "MLflow runs retrieved successfully: %d",
    len(runs)
)

# Select the columns needed for model-version comparison
display_columns = [
    column
    for column in [
        "run_id",
        "tags.mlflow.runName",
        "status",
        "metrics.mae",
        "metrics.r2"
    ]
    if column in runs.columns
]

model_version_summary = runs[
    display_columns
].copy()

model_version_summary

2026-10-08 01:39:34,216 - INFO - task6_mlops - MLflow tracking URI configured: sqlite:///C:/Users/tanak/Documents/GitHub/smart-city-traffic-capstone/part3_machine_learning/mlflow_tracking.db
2026-10-08 01:39:34,227 - INFO - task6_mlops - MLflow experiments found: 2
2026-10-08 01:39:34,264 - INFO - task6_mlops - MLflow runs retrieved successfully: 3


,run_id,tags.mlflow.runName,status
0,9b06e1bd74c74c6fb7197c2c7c8ec520,random_forest_regressor,FINISHED
1,94e52dd65d26439e921503c7c9d448a2,random_forest_regressor,FAILED
2,f3cd1322587143d896b9acd97122d8fb,linear_regression_baseline,FINISHED


### 6.2.2 Select Completed Model Versions

The MLflow database contains three historical runs. One Random Forest run has a `FAILED` status and is retained in MLflow as part of the experiment history.

A failed experimental run is useful for auditability, but it should not be treated as a deployable model version. Therefore, only runs with a `FINISHED` status are included in the model-version comparison.

The completed Linear Regression and Random Forest runs are used to confirm their recorded MAE and R² values and to select the deployment candidate.

In [5]:
# ============================================================
# 6.2.2 Select Completed Model Versions
# ============================================================

# Keep completed runs only
completed_runs = runs[
    runs["status"] == "FINISHED"
].copy()

logger.info(
    "Completed MLflow runs selected: %d",
    len(completed_runs)
)

# ------------------------------------------------------------
# Inspect available MLflow metric columns
# ------------------------------------------------------------

metric_columns = [
    column
    for column in completed_runs.columns
    if column.startswith("metrics.")
]

logger.info(
    "Metric columns available: %s",
    metric_columns
)

# ------------------------------------------------------------
# Use the metric names recorded in Task 4
# ------------------------------------------------------------

MAE_COLUMN = "metrics.MAE"
R2_COLUMN = "metrics.R_squared"

required_metric_columns = {
    MAE_COLUMN,
    R2_COLUMN
}

if not required_metric_columns.issubset(
    completed_runs.columns
):
    missing_metrics = (
        required_metric_columns
        - set(completed_runs.columns)
    )

    logger.error(
        "Required MLflow metrics missing: %s",
        sorted(missing_metrics)
    )

    raise RuntimeError(
        "Required Task 4 MLflow metrics were not found."
    )

# ------------------------------------------------------------
# Create model-version summary
# ------------------------------------------------------------

model_versions = completed_runs[
    [
        "run_id",
        "tags.mlflow.runName",
        "status",
        MAE_COLUMN,
        R2_COLUMN
    ]
].copy()

model_versions = model_versions.rename(
    columns={
        "tags.mlflow.runName": "model_name",
        MAE_COLUMN: "MAE",
        R2_COLUMN: "R2"
    }
)

model_versions["MAE"] = (
    model_versions["MAE"].round(4)
)

model_versions["R2"] = (
    model_versions["R2"].round(4)
)

# ------------------------------------------------------------
# Assign meaningful model versions
# ------------------------------------------------------------

version_mapping = {
    "linear_regression_baseline": "v1",
    "random_forest_regressor": "v2"
}

model_versions["version"] = (
    model_versions["model_name"]
    .map(version_mapping)
)

# Put version first for readability
model_versions = model_versions[
    [
        "version",
        "model_name",
        "MAE",
        "R2",
        "status",
        "run_id"
    ]
]

model_versions = (
    model_versions
    .sort_values("version")
    .reset_index(drop=True)
)

logger.info(
    "Model-version summary created successfully."
)

model_versions

2026-10-08 01:39:36,998 - INFO - task6_mlops - Completed MLflow runs selected: 2
2026-10-08 01:39:37,000 - INFO - task6_mlops - Metric columns available: ['metrics.R_squared', 'metrics.MAE']
2026-10-08 01:39:37,018 - INFO - task6_mlops - Model-version summary created successfully.


,version,model_name,MAE,R2,status,run_id
0,v1,linear_regression_baseline,813.7666,0.7259,FINISHED,f3cd1322587143d896b9acd97122d8fb
1,v2,random_forest_regressor,271.3004,0.9442,FINISHED,9b06e1bd74c74c6fb7197c2c7c8ec520


### Interpretation of Model Versions

The completed MLflow runs confirm two regression model versions from the earlier experiment.

**Version v1 – Linear Regression** achieved an MAE of **813.7666** and an R² of **0.7259**. It is retained as the baseline model.

**Version v2 – Random Forest Regressor** achieved a substantially lower MAE of **271.3004** and a higher R² of **0.9442**.

Based on these recorded MLflow metrics, **v2 Random Forest is selected as the deployment candidate** for the Task 6 deployment simulation.

The MLflow database also contains a failed Random Forest run. This run is retained as part of the experiment audit history but is excluded from model versioning and deployment because only successfully completed (`FINISHED`) runs are considered deployable candidates.

The Random Forest MAE of **271.3004** is also retained as the reference error level for the monitoring simulation later in this task.

## 6.3 Deployment Simulation with FastAPI

The selected **v2 Random Forest Regressor** is now used in a lightweight deployment simulation.

No new prediction model is trained in this section. Instead, the deployment workflow reuses the Random Forest model artifact recorded in the Task 4 MLflow experiment.

The deployment simulation consists of two stages:

1. **Load the deployed model**
   - Identify the completed Random Forest MLflow run.
   - Load its saved model artifact directly from MLflow.
   - Confirm that the model is available for inference.

2. **Expose a prediction API**
   - Create a lightweight FastAPI application.
   - Define a `/predict` endpoint.
   - Accept traffic-model input features using the same feature structure used during model training.
   - Return a predicted traffic volume.

This is a local deployment simulation rather than a production service. Its purpose is to demonstrate how the selected MLflow-tracked model could be served through an API without retraining it.


In [6]:
# ============================================================
# 6.3.1 Load the v2 Random Forest Model from MLflow
# ============================================================

# Identify the completed Random Forest run
rf_runs = completed_runs[
    completed_runs["tags.mlflow.runName"]
    == "random_forest_regressor"
].copy()

if rf_runs.empty:
    logger.error(
        "No completed Random Forest MLflow run was found."
    )
    raise RuntimeError(
        "Random Forest deployment candidate is unavailable."
    )

RF_RUN_ID = rf_runs.iloc[0]["run_id"]

logger.info(
    "Random Forest deployment run selected: %s",
    RF_RUN_ID
)

# Inspect artifacts saved under the selected run
client = mlflow.tracking.MlflowClient()

artifacts = client.list_artifacts(
    run_id=RF_RUN_ID
)

artifact_paths = [
    artifact.path
    for artifact in artifacts
]

logger.info(
    "Top-level MLflow artifacts: %s",
    artifact_paths
)

artifact_paths

2026-10-08 01:39:41,688 - INFO - task6_mlops - Random Forest deployment run selected: 9b06e1bd74c74c6fb7197c2c7c8ec520
2026-10-08 01:39:41,715 - INFO - task6_mlops - Top-level MLflow artifacts: []


[]

### Deployment Artifact Check

The completed Random Forest MLflow run was successfully identified, but no deployable model artifact was available at the top level of the existing run.

The Task 4 experiment therefore provides the historical parameters and performance metrics used for model selection, but an additional deployment-ready model artifact is required before the model can be served through an API.

To preserve the Task 4 experiment history, the existing completed run will not be modified. Instead, Task 6 will create a separate deployment run that records the selected Random Forest model as a deployable MLflow artifact while retaining the Task 4 MAE and R² as the documented model-selection baseline.

### 6.3.2 Recover the Deployment Model Specification

The existing Random Forest run does not contain a directly accessible model artifact. Before creating a deployment-ready artifact, the model specification recorded in MLflow is retrieved.

This avoids guessing the training configuration and ensures that the deployment simulation remains consistent with the Random Forest experiment selected in Task 4.

In [7]:
# ============================================================
# 6.3.2 Recover Random Forest Parameters from MLflow
# ============================================================

rf_run = mlflow.get_run(RF_RUN_ID)

rf_parameters = rf_run.data.params
rf_metrics = rf_run.data.metrics

logger.info(
    "Random Forest parameters retrieved from MLflow: %s",
    rf_parameters
)

logger.info(
    "Random Forest metrics retrieved from MLflow: %s",
    rf_metrics
)

rf_specification = pd.DataFrame(
    {
        "parameter": list(rf_parameters.keys()),
        "value": list(rf_parameters.values())
    }
)

rf_specification

2026-10-08 01:39:45,975 - INFO - task6_mlops - Random Forest parameters retrieved from MLflow: {'algorithm': 'RandomForestRegressor', 'n_estimators': '100', 'random_state': '42', 'test_size': '0.2', 'number_of_features': '25', 'feature_scaling': 'False'}
2026-10-08 01:39:45,976 - INFO - task6_mlops - Random Forest metrics retrieved from MLflow: {'MAE': 271.3003618291683, 'R_squared': 0.9441755102169268}


,parameter,value
0,algorithm,RandomForestRegressor
1,n_estimators,100
2,random_state,42
3,test_size,0.2
4,number_of_features,25
5,feature_scaling,False


### Interpretation of the Recovered Model Specification

The MLflow run confirms the exact configuration of the selected v2 Random Forest model.

The model used **100 decision trees**, `random_state=42`, an **80/20 train-test split**, **25 predictor features**, and no feature scaling. Its recorded test performance was an MAE of **271.3004** and an R² of **0.9442**.

These values are retrieved directly from the Task 4 MLflow experiment rather than manually reconstructed. They therefore provide the authoritative configuration and performance baseline for the deployment and monitoring simulation.

The next step is to recover and validate the same 25-feature input structure before creating the deployment-ready model artifact.

In [8]:
# ============================================================
# 6.3.3 Recover the Task 4 Feature Structure
# ============================================================

from pathlib import Path
import pandas as pd

DATA_PATH = Path(
    "../part2_python/engineered_traffic_data.csv"
)

deployment_df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date_time"]
)

logger.info(
    "Engineered dataset loaded for deployment: "
    "%d rows, %d columns.",
    deployment_df.shape[0],
    deployment_df.shape[1]
)

logger.info(
    "Available engineered columns: %s",
    list(deployment_df.columns)
)

pd.DataFrame(
    {
        "column_number": range(
            1,
            len(deployment_df.columns) + 1
        ),
        "column_name": deployment_df.columns
    }
)

2026-10-08 01:39:50,715 - INFO - task6_mlops - Engineered dataset loaded for deployment: 48187 rows, 30 columns.
2026-10-08 01:39:50,716 - INFO - task6_mlops - Available engineered columns: ['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description', 'date_time', 'traffic_volume', 'hour', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'has_rain', 'has_snow', 'weather_Clear', 'weather_Clouds', 'weather_Drizzle', 'weather_Fog', 'weather_Haze', 'weather_Mist', 'weather_Rain', 'weather_Smoke', 'weather_Snow', 'weather_Squall', 'weather_Thunderstorm', 'temp_scaled', 'clouds_all_scaled', 'congestion_category']


,column_number,column_name
0,1,holiday
1,2,temp
2,3,rain_1h
3,4,snow_1h
4,5,clouds_all
5,6,weather_main
6,7,weather_description
7,8,date_time
8,9,traffic_volume
9,10,hour


### 6.3.4 Reproduce the Selected Model for Deployment

The engineered dataset contains 30 columns. To reproduce the 25-feature regression input used for the selected Random Forest model, five columns are excluded:

- `weather_main` and `weather_description`, because their one-hot encoded weather features are already available.
- `date_time`, because time information is represented by engineered numerical features.
- `traffic_volume`, because it is the regression target.
- `congestion_category`, because it is derived from `traffic_volume` and would introduce target leakage.

This leaves exactly **25 predictor features**, matching the `number_of_features=25` value recorded in the Task 4 MLflow run.

The deployment model is then reproduced using the MLflow-recorded configuration:

- `RandomForestRegressor`
- `n_estimators=100`
- `random_state=42`
- `test_size=0.2`
- No feature scaling

Before the reproduced model is accepted as the deployment artifact, its MAE and R² must match the Task 4 MLflow baseline within a small numerical tolerance.

In [20]:
# ============================================================
# 6.3.4 Reproduce the v2 Random Forest Model for Deployment
# ============================================================

import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

# ------------------------------------------------------------
# 1. Convert holiday information to a numeric flag
# ------------------------------------------------------------

deployment_df["is_holiday"] = (
    deployment_df["holiday"].notna()
).astype(int)

logger.info(
    "Numeric holiday flag created: is_holiday."
)

# ------------------------------------------------------------
# 2. Reconstruct the 25-feature input structure
# ------------------------------------------------------------

excluded_columns = [
    "holiday",
    "weather_main",
    "weather_description",
    "date_time",
    "traffic_volume",
    "congestion_category"
]

FEATURE_COLUMNS = [
    column
    for column in deployment_df.columns
    if column not in excluded_columns
]

TARGET_COLUMN = "traffic_volume"

logger.info(
    "Deployment feature count: %d",
    len(FEATURE_COLUMNS)
)

if len(FEATURE_COLUMNS) != 25:

    logger.error(
        "Expected 25 deployment features, found %d.",
        len(FEATURE_COLUMNS)
    )

    raise RuntimeError(
        "Deployment feature structure does not match "
        "the Task 4 model specification."
    )

# ------------------------------------------------------------
# 3. Validate feature data types
# ------------------------------------------------------------

X = deployment_df[
    FEATURE_COLUMNS
].copy()

y = deployment_df[
    TARGET_COLUMN
].copy()

non_numeric_features = (
    X.select_dtypes(
        exclude=[np.number]
    )
    .columns
    .tolist()
)

if non_numeric_features:

    logger.error(
        "Non-numeric deployment features found: %s",
        non_numeric_features
    )

    raise RuntimeError(
        "Deployment feature matrix contains "
        "non-numeric data."
    )

logger.info(
    "Deployment feature structure validated: "
    "%d numeric features.",
    len(FEATURE_COLUMNS)
)

# ------------------------------------------------------------
# 4. Reproduce the Task 4 train-test split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)

logger.info(
    "Deployment split created: train=%d, test=%d.",
    len(X_train),
    len(X_test)
)

# ------------------------------------------------------------
# 5. Reconstruct the selected v2 Random Forest
# ------------------------------------------------------------

deployment_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=1
)

deployment_model.fit(
    X_train,
    y_train
)

logger.info(
    "Deployment Random Forest trained successfully."
)

# ------------------------------------------------------------
# 6. Evaluate the reconstructed deployment model
# ------------------------------------------------------------

deployment_predictions = (
    deployment_model.predict(
        X_test
    )
)

deployment_mae = mean_absolute_error(
    y_test,
    deployment_predictions
)

deployment_r2 = r2_score(
    y_test,
    deployment_predictions
)

logger.info(
    "Reconstructed Random Forest performance: "
    "MAE=%.4f, R2=%.4f",
    deployment_mae,
    deployment_r2
)

# ------------------------------------------------------------
# 7. Retrieve authoritative Task 4 baseline metrics
# ------------------------------------------------------------

TASK4_MAE = float(
    rf_metrics["MAE"]
)

TASK4_R2 = float(
    rf_metrics["R_squared"]
)

logger.info(
    "Task 4 baseline retained for comparison: "
    "MAE=%.4f, R2=%.4f",
    TASK4_MAE,
    TASK4_R2
)

logger.info(
    "6.3.4 model reconstruction completed successfully."
)

2026-10-08 02:13:11,775 - INFO - task6_mlops - Numeric holiday flag created: is_holiday.
2026-10-08 02:13:11,778 - INFO - task6_mlops - Deployment feature count: 25
2026-10-08 02:13:11,993 - INFO - task6_mlops - Deployment feature structure validated: 25 numeric features.
2026-10-08 02:13:12,061 - INFO - task6_mlops - Deployment split created: train=38549, test=9638.
2026-10-08 02:13:42,578 - INFO - task6_mlops - Deployment Random Forest trained successfully.
2026-10-08 02:13:43,138 - INFO - task6_mlops - Reconstructed Random Forest performance: MAE=270.0103, R2=0.9448
2026-10-08 02:13:43,140 - INFO - task6_mlops - Task 4 baseline retained for comparison: MAE=271.3004, R2=0.9442
2026-10-08 02:13:43,142 - INFO - task6_mlops - 6.3.4 model reconstruction completed successfully.


In [21]:
# ============================================================
# 6.3.5 Validate Reconstructed Deployment Model
# ============================================================

# Compare the reconstructed deployment model with the
# authoritative Task 4 MLflow baseline.

TASK4_MAE = float(rf_metrics["MAE"])
TASK4_R2 = float(rf_metrics["R_squared"])

mae_difference = deployment_mae - TASK4_MAE
r2_difference = deployment_r2 - TASK4_R2

mae_difference_pct = (
    abs(mae_difference)
    / TASK4_MAE
    * 100
)

logger.info(
    "Task 4 baseline comparison: "
    "MAE difference=%.4f (%.2f%%), "
    "R2 difference=%.4f.",
    mae_difference,
    mae_difference_pct,
    r2_difference
)

# The original Task 4 MLflow run did not retain an accessible
# deployable model artifact. Therefore, Task 6 reconstructs the
# deployment candidate from the MLflow-recorded configuration.
#
# A small performance tolerance is used only to confirm that
# the reconstructed deployment candidate behaves consistently
# with the selected Task 4 model.

MAX_MAE_DIFFERENCE_PCT = 5.0
MAX_R2_DIFFERENCE = 0.01

deployment_reproduction_valid = (
    mae_difference_pct < MAX_MAE_DIFFERENCE_PCT
    and abs(r2_difference) < MAX_R2_DIFFERENCE
)

if deployment_reproduction_valid:

    logger.info(
        "PASS - Reconstructed deployment model performance "
        "is consistent with the Task 4 baseline."
    )

else:

    logger.error(
        "FAIL - Reconstructed deployment model differs "
        "materially from the Task 4 baseline."
    )

    raise RuntimeError(
        "Deployment model performance is not sufficiently "
        "consistent with the Task 4 baseline."
    )

# Record the deployment specification for later use
deployment_specification = {
    "algorithm": "RandomForestRegressor",
    "n_estimators": 100,
    "random_state": 42,
    "test_size": 0.2,
    "number_of_features": len(FEATURE_COLUMNS),
    "feature_scaling": False,
    "task4_baseline_MAE": round(TASK4_MAE, 4),
    "task4_baseline_R2": round(TASK4_R2, 4),
    "deployment_MAE": round(deployment_mae, 4),
    "deployment_R2": round(deployment_r2, 4),
    "MAE_difference_pct": round(mae_difference_pct, 2)
}

logger.info(
    "Deployment specification confirmed: %s",
    deployment_specification
)

2026-10-08 02:14:36,113 - INFO - task6_mlops - Task 4 baseline comparison: MAE difference=-1.2900 (0.48%), R2 difference=0.0006.
2026-10-08 02:14:36,121 - INFO - task6_mlops - PASS - Reconstructed deployment model performance is consistent with the Task 4 baseline.
2026-10-08 02:14:36,123 - INFO - task6_mlops - Deployment specification confirmed: {'algorithm': 'RandomForestRegressor', 'n_estimators': 100, 'random_state': 42, 'test_size': 0.2, 'number_of_features': 25, 'feature_scaling': False, 'task4_baseline_MAE': 271.3004, 'task4_baseline_R2': 0.9442, 'deployment_MAE': 270.0103, 'deployment_R2': 0.9448, 'MAE_difference_pct': 0.48}


### 6.3.6 Create a Deployment-Ready MLflow Model Artifact

The original completed Task 4 Random Forest run retained its parameters and performance metrics but did not expose an accessible deployable model artifact.

Task 6 therefore creates a separate deployment run rather than modifying the historical Task 4 experiment.

The reconstructed Random Forest deployment candidate is recorded in MLflow together with:

- The selected model configuration
- The authoritative Task 4 baseline metrics
- The reconstructed deployment metrics
- The 25-feature input structure
- The deployable scikit-learn model artifact

This preserves the original experiment history while creating a traceable model artifact that can be loaded for the FastAPI deployment simulation.


In [13]:
# ============================================================
# 6.3.6 Create Deployment-Ready MLflow Model Artifact
# ============================================================

import mlflow.sklearn
from pathlib import Path

DEPLOYMENT_EXPERIMENT_NAME = (
    "Traffic_Volume_Deployment_Simulation"
)

mlflow.set_experiment(
    DEPLOYMENT_EXPERIMENT_NAME
)

logger.info(
    "Deployment MLflow experiment selected: %s",
    DEPLOYMENT_EXPERIMENT_NAME
)

with mlflow.start_run(
    run_name="random_forest_v2_deployment"
) as deployment_run:

    # --------------------------------------------------------
    # Log model configuration
    # --------------------------------------------------------

    mlflow.log_param(
        "algorithm",
        "RandomForestRegressor"
    )

    mlflow.log_param(
        "n_estimators",
        100
    )

    mlflow.log_param(
        "random_state",
        42
    )

    mlflow.log_param(
        "test_size",
        0.2
    )

    mlflow.log_param(
        "number_of_features",
        len(FEATURE_COLUMNS)
    )

    mlflow.log_param(
        "feature_scaling",
        False
    )

    mlflow.log_param(
        "source_task4_run_id",
        RF_RUN_ID
    )

    # --------------------------------------------------------
    # Log Task 4 baseline metrics
    # --------------------------------------------------------

    mlflow.log_metric(
        "task4_baseline_MAE",
        TASK4_MAE
    )

    mlflow.log_metric(
        "task4_baseline_R_squared",
        TASK4_R2
    )

    # --------------------------------------------------------
    # Log reconstructed deployment metrics
    # --------------------------------------------------------

    mlflow.log_metric(
        "deployment_MAE",
        deployment_mae
    )

    mlflow.log_metric(
        "deployment_R_squared",
        deployment_r2
    )

    mlflow.log_metric(
        "MAE_difference_pct",
        mae_difference_pct
    )

    # --------------------------------------------------------
    # Record exact feature structure
    # --------------------------------------------------------

    feature_file = Path(
        "deployment_features.txt"
    )

    feature_file.write_text(
        "\n".join(FEATURE_COLUMNS),
        encoding="utf-8"
    )

    mlflow.log_artifact(
        str(feature_file),
        artifact_path="metadata"
    )

    # --------------------------------------------------------
    # Log deployment-ready Random Forest model
    # --------------------------------------------------------
    #
    # The model was created locally in this notebook.
    # Only the specific sklearn tree type required by the
    # Random Forest is explicitly trusted.
    # --------------------------------------------------------

    mlflow.sklearn.log_model(
        sk_model=deployment_model,
        name="random_forest_model",
        skops_trusted_types=[
            "sklearn.tree._tree.Tree"
        ]
    )

    DEPLOYMENT_RUN_ID = (
        deployment_run.info.run_id
    )

logger.info(
    "Deployment-ready MLflow run completed: %s",
    DEPLOYMENT_RUN_ID
)

logger.info(
    "Random Forest deployment artifact logged successfully."
)

2026-10-08 01:52:55,652 - INFO - task6_mlops - Deployment MLflow experiment selected: Traffic_Volume_Deployment_Simulation
2026-10-08 01:53:58,539 - INFO - task6_mlops - Deployment-ready MLflow run completed: b22a5744718247a0a6d5c2ebda28efa7
2026-10-08 01:53:58,541 - INFO - task6_mlops - Random Forest deployment artifact logged successfully.


### 6.3.7 Verify the Deployment Model Artifact

The deployment-ready Random Forest model was successfully recorded in a new MLflow run.

Before exposing the model through FastAPI, the saved artifact is loaded back from MLflow and tested with a sample observation from the held-out test data.

This verifies that the deployment artifact is not only stored successfully but can also be restored and used for inference.

In [15]:
# ============================================================
# 6.3.7 Reload and Verify the Deployment Model Artifact
# ============================================================

MODEL_URI = (
    f"runs:/{DEPLOYMENT_RUN_ID}/random_forest_model"
)

logger.info(
    "Loading deployment model from MLflow: %s",
    MODEL_URI
)

# The trusted sklearn tree type was declared when the model
# artifact was logged. The current MLflow load_model() API
# does not accept skops_trusted_types as a load-time argument.
loaded_deployment_model = mlflow.sklearn.load_model(
    MODEL_URI
)

logger.info(
    "Deployment model loaded successfully from MLflow."
)

# ------------------------------------------------------------
# Test inference using one held-out observation
# ------------------------------------------------------------

sample_input = X_test.iloc[[0]].copy()

sample_prediction = (
    loaded_deployment_model.predict(
        sample_input
    )[0]
)

sample_actual = float(
    y_test.iloc[0]
)

logger.info(
    "Deployment inference test completed: "
    "predicted=%.1f, actual=%.1f",
    sample_prediction,
    sample_actual
)

# ------------------------------------------------------------
# Validate prediction
# ------------------------------------------------------------

if not np.isfinite(sample_prediction):

    logger.error(
        "Deployment model returned an invalid prediction."
    )

    raise RuntimeError(
        "Deployment model returned an invalid prediction."
    )

logger.info(
    "PASS - MLflow deployment artifact can be "
    "loaded and used for inference."
)

2026-10-08 01:57:37,853 - INFO - task6_mlops - Loading deployment model from MLflow: runs:/b22a5744718247a0a6d5c2ebda28efa7/random_forest_model


2026-10-08 01:57:40,838 - INFO - task6_mlops - Deployment model loaded successfully from MLflow.
2026-10-08 01:57:40,962 - INFO - task6_mlops - Deployment inference test completed: predicted=4659.6, actual=4924.0
2026-10-08 01:57:40,964 - INFO - task6_mlops - PASS - MLflow deployment artifact can be loaded and used for inference.


### 6.3.8 FastAPI Prediction Endpoint

The verified Random Forest deployment artifact is now exposed through a lightweight FastAPI application.

The API provides a `/predict` endpoint that:

- Accepts the same 25 numerical features used by the deployed Random Forest model.
- Validates that all required model features are supplied.
- Preserves the exact feature order used during model training.
- Uses the Random Forest model reloaded from the MLflow deployment artifact.
- Returns the predicted hourly traffic volume together with the deployed model version.

This is a local deployment simulation rather than a public production service. No graphical user interface is required.

In [16]:
# ============================================================
# 6.3.8 FastAPI Prediction Endpoint
# ============================================================

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Dict


# ------------------------------------------------------------
# Create FastAPI application
# ------------------------------------------------------------

app = FastAPI(
    title="Smart City Traffic Volume API",
    version="1.0"
)


# ------------------------------------------------------------
# Define request schema
# ------------------------------------------------------------

class TrafficPredictionRequest(BaseModel):
    features: Dict[str, float]


# ------------------------------------------------------------
# Prediction endpoint
# ------------------------------------------------------------

@app.post("/predict")
def predict_traffic(
    request: TrafficPredictionRequest
):
    """
    Predict hourly traffic volume using the deployed
    v2 Random Forest model.
    """

    logger.info(
        "Prediction API request received."
    )

    supplied_features = set(
        request.features.keys()
    )

    required_features = set(
        FEATURE_COLUMNS
    )

    missing_features = (
        required_features
        - supplied_features
    )

    extra_features = (
        supplied_features
        - required_features
    )

    # Validate missing features
    if missing_features:

        logger.error(
            "API request missing features: %s",
            sorted(missing_features)
        )

        raise HTTPException(
            status_code=400,
            detail=(
                "Missing required features: "
                + ", ".join(
                    sorted(missing_features)
                )
            )
        )

    # Validate unexpected features
    if extra_features:

        logger.error(
            "API request contains unexpected features: %s",
            sorted(extra_features)
        )

        raise HTTPException(
            status_code=400,
            detail=(
                "Unexpected features: "
                + ", ".join(
                    sorted(extra_features)
                )
            )
        )

    # --------------------------------------------------------
    # Build model input in exact training-feature order
    # --------------------------------------------------------

    input_df = pd.DataFrame(
        [
            {
                feature: request.features[feature]
                for feature in FEATURE_COLUMNS
            }
        ]
    )

    # --------------------------------------------------------
    # Generate prediction
    # --------------------------------------------------------

    prediction = float(
        loaded_deployment_model.predict(
            input_df
        )[0]
    )

    if not np.isfinite(prediction):

        logger.error(
            "API model returned an invalid prediction."
        )

        raise HTTPException(
            status_code=500,
            detail="Model returned an invalid prediction."
        )

    logger.info(
        "Prediction API response generated: %.1f",
        prediction
    )

    return {
        "model_version": "v2",
        "model": "RandomForestRegressor",
        "predicted_traffic_volume": round(
            prediction,
            1
        )
    }


logger.info(
    "FastAPI /predict endpoint configured successfully."
)

2026-10-08 02:00:32,408 - INFO - task6_mlops - FastAPI /predict endpoint configured successfully.


### 6.3.9 Test the FastAPI Prediction Endpoint

The `/predict` endpoint is now tested using one held-out observation from the model test dataset.

The test observation is converted into the same 25-feature request structure expected by the API. The endpoint is then called locally to confirm that:

- The request is accepted.
- The deployed v2 Random Forest model performs inference.
- A predicted traffic volume is returned.
- The response identifies the deployed model and version.

This provides a functional deployment simulation without requiring a public web server.

In [17]:
# ============================================================
# 6.3.9 Test the FastAPI Prediction Endpoint
# ============================================================

# Use one held-out test observation
api_test_features = {
    feature: float(X_test.iloc[0][feature])
    for feature in FEATURE_COLUMNS
}

# Build a validated API request
api_request = TrafficPredictionRequest(
    features=api_test_features
)

# Call the prediction endpoint locally
api_response = predict_traffic(
    api_request
)

logger.info(
    "FastAPI endpoint test completed successfully."
)

logger.info(
    "API response: %s",
    api_response
)

# Validate response structure
required_response_fields = {
    "model_version",
    "model",
    "predicted_traffic_volume"
}

api_response_valid = (
    required_response_fields.issubset(
        api_response.keys()
    )
    and np.isfinite(
        api_response["predicted_traffic_volume"]
    )
)

if not api_response_valid:

    logger.error(
        "FastAPI response validation failed."
    )

    raise RuntimeError(
        "FastAPI deployment simulation failed."
    )

logger.info(
    "PASS - FastAPI /predict endpoint accepts input "
    "and returns a valid traffic-volume prediction."
)

# User-facing API response
print(api_response)

2026-10-08 02:02:31,954 - INFO - task6_mlops - Prediction API request received.
2026-10-08 02:02:32,057 - INFO - task6_mlops - Prediction API response generated: 4659.6
2026-10-08 02:02:32,068 - INFO - task6_mlops - FastAPI endpoint test completed successfully.
2026-10-08 02:02:32,070 - INFO - task6_mlops - API response: {'model_version': 'v2', 'model': 'RandomForestRegressor', 'predicted_traffic_volume': 4659.6}
2026-10-08 02:02:32,073 - INFO - task6_mlops - PASS - FastAPI /predict endpoint accepts input and returns a valid traffic-volume prediction.


{'model_version': 'v2', 'model': 'RandomForestRegressor', 'predicted_traffic_volume': 4659.6}


### Interpretation of the Deployment Simulation

The deployment simulation successfully demonstrates the operational path from the Task 4 MLflow experiment to a prediction API.

The original Task 4 Random Forest run did not retain an accessible deployable model artifact. Therefore, a deployment candidate was reconstructed using the model configuration recorded in MLflow. The reconstructed model achieved an MAE of **270.0103** and an R² of **0.9448**, closely matching the authoritative Task 4 baseline of MAE **271.3004** and R² **0.9442**.

The reconstructed model was then stored as a new deployment-ready MLflow artifact and successfully reloaded for inference.

Finally, the FastAPI `/predict` endpoint accepted the required 25-feature input and returned a valid traffic-volume prediction of **4,659.6 vehicles per hour** using model version **v2**.

This demonstrates a functional local deployment simulation. It does not represent a public or production deployment, but it shows how the MLflow-tracked Random Forest model can be exposed through an API for prediction.

## 6.4 Prediction Error Monitoring Simulation

A deployed model should be monitored to detect whether its predictive performance has deteriorated over time.

For this deployment simulation, prediction-error monitoring is based on **Mean Absolute Error (MAE)**.

The authoritative Task 4 Random Forest test MAE of **271.3004** is used as the reference baseline.

A simple operational alert threshold is defined as:

**Alert threshold = 1.5 × baseline MAE**

Therefore:

- **PASS / Normal** if monitored MAE is less than or equal to the threshold.
- **ALERT / Requires investigation** if monitored MAE exceeds the threshold.

The 1.5 multiplier is used as a transparent simulation rule rather than a production-calibrated statistical threshold. In a real deployment, the threshold should be validated using operational history, business impact and acceptable prediction error.

Two monitoring scenarios are simulated:

1. A normal scenario with prediction error remaining within the threshold.
2. A degraded scenario with prediction error exceeding the threshold.

This demonstrates how model monitoring can be converted into a simple operational alert.


In [18]:
# ============================================================
# 6.4 Prediction Error Monitoring Simulation
# ============================================================

BASELINE_MAE = TASK4_MAE
ALERT_MULTIPLIER = 1.5

ALERT_THRESHOLD = (
    BASELINE_MAE * ALERT_MULTIPLIER
)

logger.info(
    "Monitoring baseline configured: "
    "MAE=%.4f, alert threshold=%.4f.",
    BASELINE_MAE,
    ALERT_THRESHOLD
)


def monitor_prediction_error(
    monitored_mae,
    baseline_mae=BASELINE_MAE,
    alert_threshold=ALERT_THRESHOLD
):
    """
    Compare monitored MAE with the operational alert threshold.
    """

    if monitored_mae < 0:
        raise ValueError(
            "monitored_mae cannot be negative."
        )

    error_ratio = (
        monitored_mae / baseline_mae
    )

    if monitored_mae > alert_threshold:
        status = "ALERT"
        message = "Requires investigation"
    else:
        status = "PASS"
        message = "Normal"

    result = {
        "baseline_MAE": round(
            baseline_mae, 4
        ),
        "monitored_MAE": round(
            monitored_mae, 4
        ),
        "error_ratio": round(
            error_ratio, 2
        ),
        "alert_threshold": round(
            alert_threshold, 4
        ),
        "status": status,
        "message": message
    }

    logger.info(
        "Monitoring result: MAE=%.4f, "
        "ratio=%.2f, status=%s.",
        monitored_mae,
        error_ratio,
        status
    )

    return result


# ------------------------------------------------------------
# Simulated monitoring scenarios
# ------------------------------------------------------------

normal_monitoring = monitor_prediction_error(
    monitored_mae=320.0
)

alert_monitoring = monitor_prediction_error(
    monitored_mae=450.0
)

print(normal_monitoring)
print(alert_monitoring)

2026-10-08 02:07:10,855 - INFO - task6_mlops - Monitoring baseline configured: MAE=271.3004, alert threshold=406.9505.
2026-10-08 02:07:10,860 - INFO - task6_mlops - Monitoring result: MAE=320.0000, ratio=1.18, status=PASS.
2026-10-08 02:07:10,862 - INFO - task6_mlops - Monitoring result: MAE=450.0000, ratio=1.66, status=ALERT.


{'baseline_MAE': 271.3004, 'monitored_MAE': 320.0, 'error_ratio': 1.18, 'alert_threshold': 406.9505, 'status': 'PASS', 'message': 'Normal'}
{'baseline_MAE': 271.3004, 'monitored_MAE': 450.0, 'error_ratio': 1.66, 'alert_threshold': 406.9505, 'status': 'ALERT', 'message': 'Requires investigation'}


### Interpretation of Monitoring Results

The monitoring simulation demonstrates how prediction error can be compared with the historical model-performance baseline.

The Task 4 Random Forest MAE of **271.3004** is retained as the authoritative reference. Applying the simulated 1.5-times threshold produces an alert boundary of approximately **406.9505**.

In the normal scenario, the monitored MAE is **320.0**, equivalent to approximately **1.18 times** the baseline error. Because this remains below the alert threshold, the system reports **PASS / Normal**.

In the degraded scenario, the monitored MAE increases to **450.0**, approximately **1.66 times** the baseline error. Because this exceeds the threshold, the system reports **ALERT / Requires investigation**.

This threshold is intentionally simple and transparent for the deployment simulation. It demonstrates an operational monitoring mechanism rather than a statistically calibrated production threshold.

## 6.5 Operational Alerting

The monitoring output is converted into a concise operational alert so that model-health status can be understood without inspecting raw evaluation metrics.

The alerting mechanism reports one of two states:

- **PASS / Normal** – monitored prediction error remains within the defined tolerance.
- **ALERT / Requires investigation** – monitored prediction error exceeds the defined threshold.

The alert also reports the monitored MAE and threshold so that the decision remains transparent.


In [19]:
# ============================================================
# 6.5 Operational Alerting
# ============================================================

def generate_monitoring_alert(
    monitoring_result
):
    """
    Generate a plain-language operational alert from
    the prediction-error monitoring result.
    """

    status = monitoring_result["status"]
    message = monitoring_result["message"]

    monitored_mae = monitoring_result[
        "monitored_MAE"
    ]

    threshold = monitoring_result[
        "alert_threshold"
    ]

    alert_text = (
        f"{status} / {message} - "
        f"Monitored MAE: {monitored_mae:.1f}; "
        f"Alert threshold: {threshold:.1f}."
    )

    logger.info(
        "Operational monitoring alert generated: %s",
        alert_text
    )

    return alert_text


# Generate alerts for both simulated scenarios
normal_alert = generate_monitoring_alert(
    normal_monitoring
)

degraded_alert = generate_monitoring_alert(
    alert_monitoring
)

# User-facing operational alerts
print(normal_alert)
print(degraded_alert)

2026-10-08 02:10:45,159 - INFO - task6_mlops - Operational monitoring alert generated: PASS / Normal - Monitored MAE: 320.0; Alert threshold: 407.0.
2026-10-08 02:10:45,162 - INFO - task6_mlops - Operational monitoring alert generated: ALERT / Requires investigation - Monitored MAE: 450.0; Alert threshold: 407.0.


PASS / Normal - Monitored MAE: 320.0; Alert threshold: 407.0.
ALERT / Requires investigation - Monitored MAE: 450.0; Alert threshold: 407.0.


## 6.6 Limitations

The MLOps workflow demonstrates the main stages required to move a machine-learning model from experimentation toward deployment and monitoring, but several limitations remain.

### Deployment simulation rather than production deployment
The FastAPI application is demonstrated locally and is not deployed to a public production environment. Production deployment would require additional infrastructure, authentication, security controls, availability management and operational logging.

### Reconstructed deployment model
The original completed Task 4 Random Forest run retained its parameters and metrics but did not expose an accessible deployable model artifact. Task 6 therefore reconstructed the deployment candidate using the MLflow-recorded model configuration.

The reconstructed model achieved an MAE of **270.0103** and an R² of **0.9448**, compared with the Task 4 baseline MAE of **271.3004** and R² of **0.9442**. The MAE difference is approximately **0.48%**, indicating closely aligned performance, but the reconstructed model should not be described as a binary-identical copy of the original artifact.

### Monitoring simulation
Monitoring is simulated using MAE rather than continuously calculated from live production predictions and subsequently observed traffic volumes.

The Task 4 MAE of **271.3004** remains the authoritative monitoring baseline.

### Alert threshold
The alert threshold of **1.5 times the baseline MAE** is a transparent demonstration rule rather than a statistically or operationally calibrated production threshold. A real deployment should establish thresholds using historical monitoring behaviour, operational requirements and the cost of prediction errors.

### Data and model scope
The model is based on historical observations from a single traffic corridor. Predictions should therefore not be interpreted as city-wide traffic forecasts or route recommendations.

### Git metadata
Git commit, branch and repository metadata were not automatically recorded by MLflow because the Git executable was not accessible from the Python environment. This does not affect the recorded MLflow parameters, metrics or deployment model artifact.

Overall, this implementation should be interpreted as an **MLOps and deployment simulation**, not as a production traffic-management service.

## 6.7 Final Validation

The final validation checks whether the Task 6 MLOps and deployment simulation satisfies the required components.

The validation confirms:

- Model versions are documented using completed MLflow runs.
- The Random Forest v2 model is selected using recorded MAE and R² performance.
- MLflow tracks model parameters, metrics, experiments and a deployment-ready model artifact.
- The deployment model can be reloaded from MLflow and used for inference.
- A FastAPI `/predict` endpoint accepts model input and returns a traffic-volume prediction.
- Prediction-error monitoring uses the Task 4 Random Forest MAE as the authoritative baseline.
- Both normal and degraded monitoring scenarios are demonstrated.
- Operational alerting produces `PASS / Normal` and `ALERT / Requires investigation` states.

In [22]:
# ============================================================
# 6.7 Final Validation
# ============================================================

validation_results = {}

# ------------------------------------------------------------
# 1. Model versioning
# ------------------------------------------------------------

validation_results["Model versions documented"] = (
    len(model_versions) >= 2
    and {"v1", "v2"}.issubset(
        set(model_versions["version"])
    )
)

# ------------------------------------------------------------
# 2. Random Forest v2 selected
# ------------------------------------------------------------

validation_results["Random Forest v2 selected"] = (
    (
        model_versions["model_name"]
        == "random_forest_regressor"
    ).any()
    and np.isclose(
        TASK4_MAE,
        271.3003618291683,
        atol=0.01
    )
    and np.isclose(
        TASK4_R2,
        0.9441755102169268,
        atol=0.0001
    )
)

# ------------------------------------------------------------
# 3. Reconstructed deployment model validated
# ------------------------------------------------------------

validation_results[
    "Deployment model consistent with Task 4 baseline"
] = bool(
    deployment_reproduction_valid
)

# ------------------------------------------------------------
# 4. MLflow deployment run created
# ------------------------------------------------------------

deployment_run_check = mlflow.get_run(
    DEPLOYMENT_RUN_ID
)

validation_results[
    "MLflow deployment run completed"
] = (
    deployment_run_check.info.status
    == "FINISHED"
)

# ------------------------------------------------------------
# 5. Deployment model artifact reload verified
# ------------------------------------------------------------

validation_results[
    "MLflow model artifact reload verified"
] = (
    loaded_deployment_model is not None
    and np.isfinite(sample_prediction)
)

# ------------------------------------------------------------
# 6. FastAPI endpoint configured
# ------------------------------------------------------------

api_paths = {
    route.path
    for route in app.routes
}

validation_results[
    "FastAPI predict endpoint configured"
] = (
    "/predict" in api_paths
)

# ------------------------------------------------------------
# 7. FastAPI prediction response validated
# ------------------------------------------------------------

validation_results[
    "FastAPI prediction returned successfully"
] = (
    api_response_valid
    and api_response.get("model_version") == "v2"
    and np.isfinite(
        api_response.get(
            "predicted_traffic_volume",
            np.nan
        )
    )
)

# ------------------------------------------------------------
# 8. Monitoring baseline and threshold configured
# ------------------------------------------------------------

validation_results[
    "Prediction-error monitoring configured"
] = (
    np.isclose(
        BASELINE_MAE,
        TASK4_MAE
    )
    and np.isclose(
        ALERT_THRESHOLD,
        TASK4_MAE * 1.5
    )
)

# ------------------------------------------------------------
# 9. PASS monitoring state demonstrated
# ------------------------------------------------------------

validation_results[
    "PASS monitoring state demonstrated"
] = (
    normal_monitoring["status"] == "PASS"
    and normal_monitoring["message"] == "Normal"
)

# ------------------------------------------------------------
# 10. ALERT monitoring state demonstrated
# ------------------------------------------------------------

validation_results[
    "ALERT monitoring state demonstrated"
] = (
    alert_monitoring["status"] == "ALERT"
    and alert_monitoring["message"]
    == "Requires investigation"
)

# ------------------------------------------------------------
# 11. Operational alerts generated
# ------------------------------------------------------------

validation_results[
    "Operational alert messages generated"
] = (
    normal_alert.startswith(
        "PASS / Normal"
    )
    and degraded_alert.startswith(
        "ALERT / Requires investigation"
    )
)

# ------------------------------------------------------------
# Log validation results
# ------------------------------------------------------------

logger.info(
    "Task 6 final validation started."
)

for check, passed in validation_results.items():

    if passed:
        logger.info(
            "PASS - %s",
            check
        )

    else:
        logger.error(
            "FAIL - %s",
            check
        )

# ------------------------------------------------------------
# Final Task 6 status
# ------------------------------------------------------------

task6_passed = all(
    validation_results.values()
)

if task6_passed:

    logger.info(
        "TASK 6 STATUS: PASS - "
        "All required checks completed."
    )

else:

    logger.error(
        "TASK 6 STATUS: FAIL - "
        "One or more required checks failed."
    )

2026-10-08 02:17:07,384 - INFO - task6_mlops - Task 6 final validation started.
2026-10-08 02:17:07,388 - INFO - task6_mlops - PASS - Model versions documented
2026-10-08 02:17:07,390 - INFO - task6_mlops - PASS - Random Forest v2 selected
2026-10-08 02:17:07,393 - INFO - task6_mlops - PASS - Deployment model consistent with Task 4 baseline
2026-10-08 02:17:07,394 - INFO - task6_mlops - PASS - MLflow deployment run completed
2026-10-08 02:17:07,396 - INFO - task6_mlops - PASS - MLflow model artifact reload verified
2026-10-08 02:17:07,397 - INFO - task6_mlops - PASS - FastAPI predict endpoint configured
2026-10-08 02:17:07,400 - INFO - task6_mlops - PASS - FastAPI prediction returned successfully
2026-10-08 02:17:07,404 - INFO - task6_mlops - PASS - Prediction-error monitoring configured
2026-10-08 02:17:07,405 - INFO - task6_mlops - PASS - PASS monitoring state demonstrated
2026-10-08 02:17:07,407 - INFO - task6_mlops - PASS - ALERT monitoring state demonstrated
2026-10-08 02:17:07,40